In [ ]:
import json
from typing import List, Dict, Any

In [ ]:
def enrich_shap_features(
    input_data: List[Dict[str, Any]], 
    knowledge_base_path: str = "cicflowmeter_features_knowledge_base_new.json"
) -> List[Dict[str, Any]]:
    """
    Enrich SHAP feature explanations with detailed information from the knowledge base.
    """
    # Load knowledge base
    with open(knowledge_base_path, 'r', encoding='utf-8') as f:
        knowledge_base = json.load(f)
    
    # Create a lookup dict for fast feature matching (case-insensitive, flexible key matching)
    kb_lookup = {}
    for entry in knowledge_base:
        feature_key = entry.get("feature_name", "").lower().strip()
        kb_lookup[feature_key] = entry
    
    enriched = []
    
    for item in input_data:
        feature_name = item.get("feature", "")
        feature_key = feature_name.lower().strip()
        
        # Find matching entry in knowledge base
        kb_entry = kb_lookup.get(feature_key)
        
        # Calculate contribution (percentage based on absolute SHAP)
        total_abs_shap = sum(abs(d["shap"]) for d in input_data)
        contribution = item["contribution"] if total_abs_shap > 0 else 0
        
        # Determine direction
        direction = "increase" if item["shap"] > 0 else "decrease"
        
        new_item = {
            "feature": feature_name,
            "value": item["value"],
            "shap": item["shap"],
            "abs_shap": item["abs_shap"],
            "contribution": round(contribution, 2),
            "direction": direction,
        }
        
        if kb_entry:
            new_item.update({
                "category": kb_entry.get("category", "Unknown"),
                "definition": kb_entry.get("definition", ""),
                "note": kb_entry.get("note", "")
            })
        else:
            # Fallback for unknown features
            new_item.update({
                "definition": "",
                "category": ""
            })
        
        enriched.append(new_item)
    
    return enriched

In [ ]:
import shap

explainer = shap.TreeExplainer(xg)

shap_values = explainer(X_test)

In [ ]:
shap.summary_plot(shap_values, X_test)

In [ ]:
print(type(labelencoder.classes_))
cls_names = np.array(labelencoder.classes_)
print(type(cls_names))

In [ ]:
classes = {}
for i in range(7):
    classes[i] = cls_names[i]
print(classes)

In [ ]:
pd.Series(y_test).map(classes).value_counts()

In [ ]:
import numpy as np
import pandas as pd

# -----------------------------
# Configuration
# -----------------------------

classes = {
    0: 'BENIGN',
    1: 'Bot',
    2: 'BruteForce',
    3: 'DoS',
    4: 'Infiltration',
    5: 'PortScan',
    6: 'WebAttack'
}

# All attack classes (excluding BENIGN)
attack_labels = [1, 2, 3, 4, 5, 6]

# -----------------------------
# XGBoost prediction
# -----------------------------

pred_probs = xg.predict_proba(X_test)

pred_class = np.argmax(pred_probs, axis=1)

pred_conf = np.max(pred_probs, axis=1)

# Identify correctly classified samples
correct_mask = (pred_class == np.asarray(y_test))

# -----------------------------
# Select all correctly classified attack samples
# -----------------------------

experiment_mask = (
    np.isin(np.asarray(y_test), attack_labels)
    & correct_mask
)

indices = np.where(experiment_mask)[0]

experiment_rows = []

for idx in indices:
    true_label = int(y_test[idx])
    predicted_label = int(pred_class[idx])

    experiment_rows.append({
        "sample_id": int(idx),
        "ground_truth_id": true_label,
        "ground_truth_label": classes[true_label],
        "prediction_id": predicted_label,
        "prediction_label": classes[predicted_label],
        "confidence": float(pred_conf[idx])
    })

# -----------------------------
# Create DataFrame
# -----------------------------

experiment_df = pd.DataFrame(
    experiment_rows,
    columns=[
        "sample_id",
        "ground_truth_id",
        "ground_truth_label",
        "prediction_id",
        "prediction_label",
        "confidence"
    ]
)

experiment_df = experiment_df.sort_values(
    ["prediction_label", "sample_id"]
).reset_index(drop=True)

# -----------------------------
# Display results
# -----------------------------

print(experiment_df.head())

print("\nNumber of correctly classified samples per class:\n")

print(
    experiment_df["prediction_label"]
    .value_counts()
    .sort_index()
)

print("\nTotal selected samples:", len(experiment_df))

In [ ]:
import numpy as np

duplicates = {}

for _, row in experiment_df.iterrows():

    sid = int(row["sample_id"])

    sample = X_test[sid]

    key = hash(sample.tobytes())

    duplicates.setdefault(key, []).append(sid)

print("Duplicate groups:\n")

for key, ids in duplicates.items():

    if len(ids) > 1:

        print(ids)

In [ ]:
CONTRIBUTION_THRESHOLD = 1
CUMULATIVE_TARGET = 90

all_sample_results = []
feature_names = list(df.columns[:-1])

for _, row in experiment_df.iterrows():
    sample_id = int(row["sample_id"])

    sample = X_test[sample_id].reshape(1,-1)

    prediction = row["prediction_label"]

    confidence = float(row["confidence"])
    
    fixed_prediction = {
        "label": prediction,
        "confidence": confidence
    }
    
    # print(sample_id)

    # print(hash(sample.tobytes()))

    # print(sample[0][:10])
    
    shap_values = explainer(sample)
    predicted_class = int(row["prediction_id"])
    exp = shap_values[0,:,predicted_class]
    total = np.abs(exp.values).sum()
    if total == 0:
        continue
    
    # print("total:", total)

    # print(np.sort(np.abs(exp.values))[::-1][:15])
    
    features = []
    cumulative = 0.0
    
    sorted_idx = np.argsort(np.abs(exp.values))[::-1]
    
    for i in sorted_idx: 
        contribution = abs(exp.values[i]) / total * 100

        if contribution < CONTRIBUTION_THRESHOLD:
            break
        
        features.append({
            "feature": feature_names[i],
            "value": float(sample[0][i]),
            "shap": float(exp.values[i]),
            "abs_shap": float(abs(exp.values[i])),
            "contribution": round(contribution,2),
            "direction": "increase" if exp.values[i] > 0 else "decrease"
        })
        
        cumulative += contribution
        if cumulative >= CUMULATIVE_TARGET:
            break
    
    top_features_tree_shap_threshold = features
    top_features_with_info = enrich_shap_features(top_features_tree_shap_threshold)
    sample_result = {
        "sample_id": sample_id,
        "ground_truth": row["ground_truth_label"],
        "prediction": fixed_prediction,
        "all_shap_values": [float(v) for v in exp.values],
        "top_features_with_info": top_features_with_info
    }
    
    all_sample_results.append(sample_result)
    
    # print(sample_result)
print(len(all_sample_results))

In [ ]:
seen = set()
unique_samples = []

for sample in all_sample_results:

    label = sample["prediction"]["label"]

    confidence = round(
        float(sample["prediction"]["confidence"]), 4
    )

    shap_values = tuple(
        round(float(v), 6)
        for v in sample["all_shap_values"]
    )

    key = (
        label,
        confidence,
        shap_values
    )

    if key not in seen:
        seen.add(key)
        unique_samples.append(sample)

print("Before:", len(all_sample_results))
print("After:", len(unique_samples))
print("Removed:", len(all_sample_results) - len(unique_samples))

In [ ]:
import pandas as pd

class_counts_df = pd.Series(
    [sample["prediction"]["label"] for sample in unique_samples]
).value_counts().rename_axis("Class").reset_index(name="Count")

print(class_counts_df)

In [ ]:
import numpy as np
import pandas as pd

# =========================================================
# Configuration
# =========================================================

TARGET_PER_CLASS = {
    "WebAttack": 49,
    "BruteForce": 36,
    "DoS": 231,
    "PortScan": 26,
    "Bot": 53,
    "Infiltration": 5
}

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


# =========================================================
# Step 1: Build metadata
# =========================================================

metadata = []

for idx, sample in enumerate(unique_samples):

    label = sample["prediction"]["label"]
    confidence = float(sample["prediction"]["confidence"])

    # -----------------------------
    # Confidence bin
    # -----------------------------

    if confidence < 0.60:
        confidence_bin = "0.00-0.59"

    elif confidence < 0.70:
        confidence_bin = "0.60-0.69"

    elif confidence < 0.80:
        confidence_bin = "0.70-0.79"

    elif confidence < 0.90:
        confidence_bin = "0.80-0.89"

    elif confidence < 0.95:
        confidence_bin = "0.90-0.94"

    else:
        confidence_bin = "0.95-1.00"


    # -----------------------------
    # SHAP signature
    # -----------------------------
    
    shap_signature = tuple(
        (item["feature"], item["direction"])
        for item in sample["top_features_with_info"]
    )


    metadata.append({
        "index": idx,
        "label": label,
        "confidence": confidence,
        "confidence_bin": confidence_bin,
        "shap_signature": shap_signature
    })


metadata_df = pd.DataFrame(metadata)


# =========================================================
# Step 2: Select samples
#         Class -> Confidence -> SHAP diversity
# =========================================================

selected_indices = []

confidence_order = [
    "0.00-0.59",
    "0.60-0.69",
    "0.70-0.79",
    "0.80-0.89",
    "0.90-0.94",
    "0.95-1.00"
]


for label, target_n in TARGET_PER_CLASS.items():

    class_df = metadata_df[
        metadata_df["label"] == label
    ].copy()

    available_n = len(class_df)

    # Safety check
    if available_n < target_n:
        print(
            f"WARNING: {label}: requested {target_n}, "
            f"but only {available_n} available."
        )
        target_n = available_n


    # -----------------------------------------------------
    # Create groups:
    # confidence bin -> SHAP signature
    # -----------------------------------------------------

    confidence_groups = {}

    for confidence_bin in confidence_order:

        bin_df = class_df[
            class_df["confidence_bin"] == confidence_bin
        ]

        if len(bin_df) == 0:
            continue

        shap_groups = []

        for _, group in bin_df.groupby(
            "shap_signature",
            sort=False
        ):

            indices = group["index"].tolist()

            # Randomize samples inside each SHAP group
            rng.shuffle(indices)

            shap_groups.append(indices)


        # Randomize order of SHAP groups
        rng.shuffle(shap_groups)

        confidence_groups[confidence_bin] = shap_groups


    # -----------------------------------------------------
    # Round-robin across confidence bins
    # and SHAP groups
    # -----------------------------------------------------

    class_selected = []

    while len(class_selected) < target_n:

        progress = False

        for confidence_bin in confidence_order:

            if confidence_bin not in confidence_groups:
                continue

            shap_groups = confidence_groups[confidence_bin]

            # Remove empty groups
            shap_groups = [
                group for group in shap_groups
                if len(group) > 0
            ]

            confidence_groups[confidence_bin] = shap_groups

            if len(shap_groups) == 0:
                continue

            # Select one sample from one SHAP pattern
            group = shap_groups[0]

            selected_idx = group.pop()

            class_selected.append(selected_idx)

            progress = True

            if len(class_selected) == target_n:
                break

            # Rotate this SHAP group to the end
            shap_groups.append(shap_groups.pop(0))


        if not progress:
            break


    selected_indices.extend(class_selected)


# =========================================================
# Step 3: Create final selected sample set
# =========================================================

selected_samples = [
    unique_samples[i]
    for i in selected_indices
]

# Randomize final order
rng.shuffle(selected_samples)


# =========================================================
# Step 4: Report class distribution
# =========================================================

print("=" * 60)
print("FINAL SAMPLE SELECTION")
print("=" * 60)

print("\nOriginal unique samples:", len(unique_samples))
print("Selected samples:", len(selected_samples))


print("\nSelected samples per class:\n")

selected_class_counts = pd.Series(
    [
        sample["prediction"]["label"]
        for sample in selected_samples
    ]
).value_counts()

print(selected_class_counts)


# =========================================================
# Step 5: Report confidence distribution
# =========================================================

selected_confidences = [
    float(sample["prediction"]["confidence"])
    for sample in selected_samples
]

selected_confidence_bins = pd.cut(
    selected_confidences,
    bins=[0, 0.60, 0.70, 0.80, 0.90, 0.95, 1.0],
    labels=confidence_order,
    include_lowest=True
)

print("\nConfidence distribution:\n")

print(
    pd.Series(selected_confidence_bins)
    .value_counts()
    .reindex(confidence_order, fill_value=0)
)


# =========================================================
# Step 6: Confidence distribution by class
# =========================================================

print("\nConfidence distribution by class:\n")

confidence_by_class = pd.crosstab(
    [
        sample["prediction"]["label"]
        for sample in selected_samples
    ],
    selected_confidence_bins
)

confidence_by_class = confidence_by_class.reindex(
    index=TARGET_PER_CLASS.keys(),
    columns=confidence_order,
    fill_value=0
)

print(confidence_by_class)

In [ ]:
# =========================================================
# Configuration
# =========================================================

TARGET_PER_CLASS = {
    "WebAttack": 41,
    "BruteForce": 74,
    "DoS": 100,
    "Bot": 100,
    "Infiltration": 100
}

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


# =========================================================
# Step 1: Build metadata
# =========================================================

metadata = []

for idx, sample in enumerate(unique_samples):

    label = sample["prediction"]["label"]
    confidence = float(sample["prediction"]["confidence"])

    # -----------------------------
    # Confidence bin
    # -----------------------------

    if confidence < 0.60:
        confidence_bin = "0.00-0.59"

    elif confidence < 0.70:
        confidence_bin = "0.60-0.69"

    elif confidence < 0.80:
        confidence_bin = "0.70-0.79"

    elif confidence < 0.90:
        confidence_bin = "0.80-0.89"

    elif confidence < 0.95:
        confidence_bin = "0.90-0.94"

    else:
        confidence_bin = "0.95-1.00"


    # -----------------------------
    # SHAP signature
    # -----------------------------
    
    shap_signature = tuple(
        (item["feature"], item["direction"])
        for item in sample["top_features_with_info"]
    )


    metadata.append({
        "index": idx,
        "label": label,
        "confidence": confidence,
        "confidence_bin": confidence_bin,
        "shap_signature": shap_signature
    })


metadata_df = pd.DataFrame(metadata)


# =========================================================
# Step 2: Select samples
#         Class -> Confidence -> SHAP diversity
# =========================================================

selected_indices = []

confidence_order = [
    "0.00-0.59",
    "0.60-0.69",
    "0.70-0.79",
    "0.80-0.89",
    "0.90-0.94",
    "0.95-1.00"
]


for label, target_n in TARGET_PER_CLASS.items():

    class_df = metadata_df[
        metadata_df["label"] == label
    ].copy()

    available_n = len(class_df)

    # Safety check
    if available_n < target_n:
        print(
            f"WARNING: {label}: requested {target_n}, "
            f"but only {available_n} available."
        )
        target_n = available_n


    # -----------------------------------------------------
    # Create groups:
    # confidence bin -> SHAP signature
    # -----------------------------------------------------

    confidence_groups = {}

    for confidence_bin in confidence_order:

        bin_df = class_df[
            class_df["confidence_bin"] == confidence_bin
        ]

        if len(bin_df) == 0:
            continue

        shap_groups = []

        for _, group in bin_df.groupby(
            "shap_signature",
            sort=False
        ):

            indices = group["index"].tolist()

            # Randomize samples inside each SHAP group
            rng.shuffle(indices)

            shap_groups.append(indices)


        # Randomize order of SHAP groups
        rng.shuffle(shap_groups)

        confidence_groups[confidence_bin] = shap_groups


    # -----------------------------------------------------
    # Round-robin across confidence bins
    # and SHAP groups
    # -----------------------------------------------------

    class_selected = []

    while len(class_selected) < target_n:

        progress = False

        for confidence_bin in confidence_order:

            if confidence_bin not in confidence_groups:
                continue

            shap_groups = confidence_groups[confidence_bin]

            # Remove empty groups
            shap_groups = [
                group for group in shap_groups
                if len(group) > 0
            ]

            confidence_groups[confidence_bin] = shap_groups

            if len(shap_groups) == 0:
                continue

            # Select one sample from one SHAP pattern
            group = shap_groups[0]

            selected_idx = group.pop()

            class_selected.append(selected_idx)

            progress = True

            if len(class_selected) == target_n:
                break

            # Rotate this SHAP group to the end
            shap_groups.append(shap_groups.pop(0))


        if not progress:
            break


    selected_indices.extend(class_selected)


# =========================================================
# Step 3: Create final selected sample set
# =========================================================

selected_samples = [
    unique_samples[i]
    for i in selected_indices
]

# Randomize final order
rng.shuffle(selected_samples)


# =========================================================
# Step 4: Report class distribution
# =========================================================

print("=" * 60)
print("FINAL SAMPLE SELECTION")
print("=" * 60)

print("\nOriginal unique samples:", len(unique_samples))
print("Selected samples:", len(selected_samples))


print("\nSelected samples per class:\n")

selected_class_counts = pd.Series(
    [
        sample["prediction"]["label"]
        for sample in selected_samples
    ]
).value_counts()

print(selected_class_counts)


# =========================================================
# Step 5: Report confidence distribution
# =========================================================

selected_confidences = [
    float(sample["prediction"]["confidence"])
    for sample in selected_samples
]

selected_confidence_bins = pd.cut(
    selected_confidences,
    bins=[0, 0.60, 0.70, 0.80, 0.90, 0.95, 1.0],
    labels=confidence_order,
    include_lowest=True
)

print("\nConfidence distribution:\n")

print(
    pd.Series(selected_confidence_bins)
    .value_counts()
    .reindex(confidence_order, fill_value=0)
)


# =========================================================
# Step 6: Confidence distribution by class
# =========================================================

print("\nConfidence distribution by class:\n")

confidence_by_class = pd.crosstab(
    [
        sample["prediction"]["label"]
        for sample in selected_samples
    ],
    selected_confidence_bins
)

confidence_by_class = confidence_by_class.reindex(
    index=TARGET_PER_CLASS.keys(),
    columns=confidence_order,
    fill_value=0
)

print(confidence_by_class)

In [ ]:
import json

# ============================================
# Configuration
# ============================================

JSON_FILE = "final_results_1.json"


# ============================================
# Step 1: Read JSON and extract sample_ids
# ============================================

with open(JSON_FILE, "r", encoding="utf-8") as f:
    json_data = json.load(f)

excluded_ids = {
    int(item["sample_id"])
    for item in json_data
}

print("Number of sample IDs in JSON:", len(excluded_ids))


# ============================================
# Step 2: Remove these samples from
#          the final CICIDS2017 dataset
# ============================================

before_count = len(selected_samples)

filtered_samples = [
    sample
    for sample in selected_samples
    if int(sample["sample_id"]) not in excluded_ids
]


# ============================================
# Step 3: Report results
# ============================================

after_count = len(filtered_samples)
removed_count = before_count - after_count

print("\nBefore:", before_count)
print("After:", after_count)
print("Removed:", removed_count)

In [ ]:
print(len(selected_samples))

In [ ]:
print(selected_samples[0])

In [ ]:
# Remove all_shap_values from each selected sample
for sample in selected_samples:
    sample.pop("all_shap_values", None)

print("all_shap_values removed from all selected samples.")

In [ ]:
def create_llm_input(sample):

    return {

        "sample_id":
            sample["sample_id"],


        "freezed_prediction":
            sample["prediction"],


        "top_features":
            sample["top_features_with_info"]

    }

In [ ]:
def convert_numpy(obj):

    if isinstance(obj, np.integer):
        return int(obj)

    elif isinstance(obj, np.floating):
        return float(obj)

    elif isinstance(obj, np.ndarray):
        return obj.tolist()

    elif isinstance(obj, dict):
        return {
            k: convert_numpy(v)
            for k, v in obj.items()
        }

    elif isinstance(obj, list):
        return [
            convert_numpy(i)
            for i in obj
        ]

    elif isinstance(obj, tuple):
        return tuple(
            convert_numpy(i)
            for i in obj
        )

    else:
        return obj

In [ ]:
clean_samples = convert_numpy(selected_samples)